In [ ]:
import requests
import csv
import os
import threading
import hashlib
from concurrent.futures import ThreadPoolExecutor, as_completed

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

lock = threading.Lock()
completed = 0
success_count = 0
fail_count = 0

CSV_FILE = "deface_labels.csv"

def load_urls(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

# def create_filename(url):
#     return hashlib.md5(url.encode()).hexdigest() + ".html"
def create_filename(url):
    filename = url.replace("http://", "").replace("https://", "")
    filename = filename.replace("/", "_")
    filename = filename.replace("?", "_").replace("&", "_").replace("=", "_")

    return filename[:200] + ".html"

def fetch_html(url, retries=3):
    for _ in range(retries):
        try:
            res = requests.get(url, headers=HEADERS, timeout=(3,5))
            if res.status_code == 200:
                return url, res.text
        except:
            pass
    return url, None

def load_existing_urls():
    if not os.path.exists(CSV_FILE):
        return set()

    urls = set()
    with open(CSV_FILE, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader, None)
        for row in reader:
            urls.add(row[0])
    return urls

def append_csv(row):
    file_exists = os.path.exists(CSV_FILE)

    with open(CSV_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["URL", "HTML_File_Name", "Label"])
        writer.writerow(row)

def save_result(result, total, existing_urls):
    global completed, success_count, fail_count

    url, html = result

    with lock:
        completed += 1
        percent = (completed / total) * 100

    if url in existing_urls:
        print(f"[SKIP] {url}")
        return None

    if not html:
        with lock:
            fail_count += 1
        print(f"[{completed}/{total} - {percent:.1f}%] FAILED: {url}")
        return None

    filename = create_filename(url)
    path = os.path.join("html_pages", filename)

    with open(path, "w", encoding="utf-8") as f:
        f.write(html)

    with lock:
        success_count += 1

    append_csv([url, filename, "unlabeled"])

    print(f"[{completed}/{total} - {percent:.1f}%] OK: {url}")
    return True

def main():
    urls = load_urls("urls_alive.txt")
    total = len(urls)

    os.makedirs("html_pages", exist_ok=True)

    existing_urls = load_existing_urls()

    num_threads = 10

    print(f"Total URLs: {total}")
    print(f"Running with {num_threads} threads...\n")

    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        futures = [executor.submit(fetch_html, url) for url in urls]

        for future in as_completed(futures):
            save_result(future.result(), total, existing_urls)

    print("\n===== SUMMARY =====")
    print(f"Total: {total}")
    print(f"Success: {success_count}")
    print(f"Failed: {fail_count}")

    print("\nDone!")

if __name__ == "__main__":
    main()

Total URLs: 27
Running with 10 threads...

[1/27 - 3.7%] OK: http://www.kejari-sleman.go.id
[2/27 - 7.4%] OK: http://commandcenter.bimakota.go.id
[3/27 - 11.1%] OK: http://behalinternational.com
[4/27 - 14.8%] OK: http://twoguysandabucket.com
[5/27 - 18.5%] FAILED: http://alternativaradio.fm
[6/27 - 22.2%] OK: http://siggec.gov.ao
[7/27 - 25.9%] OK: http://rompetesgrow.com.ar
[8/27 - 29.6%] FAILED: http://apps.telessaude.hc.ufmg.br
[9/27 - 33.3%] FAILED: http://faminvestment.ae
[10/27 - 37.0%] OK: http://revistas.face.ufmg.br
[11/27 - 40.7%] OK: http://fastbooks.info
[12/27 - 44.4%] OK: http://elementsrealfood.com
[13/27 - 48.1%] OK: http://perfectlifestyle.info
[14/27 - 51.9%] OK: http://espb.ao
[15/27 - 55.6%] OK: http://holyspain.com
[16/27 - 59.3%] OK: http://revistas.ufrj.br
[17/27 - 63.0%] FAILED: http://network.hr
[18/27 - 66.7%] OK: http://periodicos.ufal.br
[19/27 - 70.4%] OK: http://solusiherbalalami.com
[20/27 - 74.1%] OK: http://thebloomingstoryindia.com
[21/27 - 77.8%] OK: